# File for training the GCN model
This file will be used to train the Graph Convolution Model

In [1]:
!pip install pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
     ------- -------------------------------- 0.8/4.0 MB 3.4 MB/s eta 0:00:01
     -------------------------- ------------- 2.6/4.0 MB 5.2 MB/s eta 0:00:01
     ---------------------------------------- 4.0/4.0 MB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for torch_scatter: filename=torch_scatter-2.1.2-cp311-cp311-win_amd64.whl size=604436 sha256=


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\thijn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Below a small GCN model will be trained on the data set

First we split the data 

In [ ]:
from torch_geometric.data import Data
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("transaction_dataset.csv")

target_col = "FLAG"
if target_col not in data.columns:
    raise ValueError("Expected target column 'FLAG' in transaction_dataset.csv")

# Drop obvious leakage / identifier columns if present.
leakage_cols = ["Unnamed: 0", "Index"]
feature_df = data.drop(columns=[target_col] + leakage_cols, errors="ignore")

# Keep only numeric features for the GCN input.
num_df = feature_df.select_dtypes(include="number")
if num_df.shape[1] == 0:
    X = torch.zeros((len(data), 1), dtype=torch.float)
else:
    y_array = data[target_col].to_numpy()
    all_idx = np.arange(len(data))
    train_idx_np, test_idx_np = train_test_split(
        all_idx,
        test_size=0.2,
        random_state=42,
        stratify=y_array,
    )

    # Fit preprocessing on train rows only to avoid test leakage.
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    train_features = imputer.fit_transform(num_df.iloc[train_idx_np])
    train_features = scaler.fit_transform(train_features)
    all_features = scaler.transform(imputer.transform(num_df))

    X = torch.tensor(all_features, dtype=torch.float)

    train_idx = torch.tensor(train_idx_np, dtype=torch.long)
    test_idx = torch.tensor(test_idx_np, dtype=torch.long)

# Build a simple valid edge_index (chain graph: 0->1->2->...)
num_nodes = X.size(0)
if num_nodes >= 2:
    src = torch.arange(0, num_nodes - 1, dtype=torch.long)
    dst = torch.arange(1, num_nodes, dtype=torch.long)
    edge_index = torch.stack([src, dst], dim=0)
else:
    edge_index = torch.empty((2, 0), dtype=torch.long)

# Add labels
y = torch.tensor(data[target_col].values, dtype=torch.long)

# Build graph data object
train_data = Data(x=X, edge_index=edge_index, y=y)

# Train/test split masks
num_nodes = train_data.num_nodes
train_data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_data.train_mask[train_idx] = True
train_data.test_mask[test_idx] = True

train_pos = int(train_data.y[train_data.train_mask].sum().item())
train_total = int(train_data.train_mask.sum().item())
test_pos = int(train_data.y[train_data.test_mask].sum().item())
test_total = int(train_data.test_mask.sum().item())
majority_baseline = max(train_total - train_pos, train_pos) / train_total if train_total > 0 else 0

print(f"Nodes: {num_nodes}, Features: {X.size(1)}, Classes: {int(y.max().item()) + 1}")
print(f"Train class-1 rate: {train_pos / train_total:.4f}, Test class-1 rate: {test_pos / test_total:.4f}")
print(f"Majority-class baseline accuracy on train split: {majority_baseline:.4f}")
print(f"Any NaN in X: {torch.isnan(X).any().item()}, Any Inf in X: {torch.isinf(X).any().item()}")

Nodes: 9841, Features: 47, Classes: 2
Any NaN in X: False, Any Inf in X: False


Training the model

In [ ]:
# train_data now contains the graph with node features, edge index, labels, and train/test masks.
# training the model below

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x  # raw logits, no log_softmax

num_classes = int(train_data.y.max().item()) + 1
model = GCN(in_channels=X.size(1), hidden_channels=16, out_channels=num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(200):
    model.train()
    optimizer.zero_grad()

    out = model(train_data)  # [num_nodes, num_classes]
    logits = out[train_data.train_mask]
    target = train_data.y[train_data.train_mask]
    loss = criterion(logits, target)

    if not torch.isfinite(loss):
        raise RuntimeError("Loss became non-finite. Check feature preprocessing and labels.")

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    out = model(train_data)
    test_logits = out[train_data.test_mask]
    test_target = train_data.y[train_data.test_mask]
    pred = test_logits.argmax(dim=1)
    correct = (pred == test_target).sum().item()
    total = test_target.numel()
    accuracy = correct / total if total > 0 else 0
    print(f"Test Accuracy: {accuracy:.4f}")
    if total > 0:
        probs = torch.softmax(test_logits, dim=1)[:, 1]
        print(f"Mean predicted probability for class 1: {probs.mean().item():.4f}")

Epoch 20, Loss: 0.3474
Epoch 40, Loss: 0.2140
Epoch 60, Loss: 0.1359
Epoch 80, Loss: 0.0964
Epoch 100, Loss: 0.0731
Epoch 120, Loss: 0.0633
Epoch 140, Loss: 0.0518
Epoch 160, Loss: 0.0451
Epoch 180, Loss: 0.0411
Epoch 200, Loss: 0.0343
Test Accuracy: 0.9995
